# 10 - Advanced Model Training

## Context
Setelah memahami performa baseline, kita sekarang melatih model yang lebih kompleks menggunakan algoritma gradient boosting: LightGBM, XGBoost, dan CatBoost. Kita menggunakan strategi 5-Fold Cross Validation untuk mendapatkan estimasi performa model yang stabil dan tidak bias, serta membandingkan performa akhir dan feature importance masing-masing model.

## Objective
- Melatih LightGBM, XGBoost, dan CatBoost dengan Stratified 5-Fold Cross Validation.
- Membandingkan skor evaluasi (ROC-AUC) antar algoritma.
- Menganalisis dan membandingkan peringkat feature importance yang dihasilkan.

### Impor Library

In [ ]:
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

### Muat Dataset Terproses

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")

train = pd.read_parquet(path)
print(f"Data berhasil dimuat: {train.shape}")
print(f"Fraud rate: {train['isFraud'].mean()*100:.2f}%")

### Persiapan Fitur & Splitter Cross Validation

In [ ]:
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT']
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

X = train[feature_cols]
y = train['isFraud']

# Gunakan StratifiedKFold untuk memastikan rasio fraud terjaga di tiap lipatan
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### LightGBM: 5-Fold Cross Validation

In [ ]:
lgb_scores = []
lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42
}

start = time.perf_counter()
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=200,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(30, verbose=False)]
    )

    preds = model.predict(X_va, num_iteration=model.best_iteration)
    lgb_scores.append(roc_auc_score(y_va, preds))

elapsed_lgb = time.perf_counter() - start
mean_lgb = np.mean(lgb_scores)
std_lgb = np.std(lgb_scores)
print(f"Rata-rata AUC: {mean_lgb:.5f} ± {std_lgb:.5f}")
print(f"Skor tiap lipatan (fold): {[round(s, 5) for s in lgb_scores]}")

### LightGBM: Pelatihan Model Final

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)

start = time.perf_counter()
model_lgb = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=200,
    valid_sets=[dtrain, dval],
    callbacks=[lgb.early_stopping(30, verbose=False)]
)
time_lgb = time.perf_counter() - start

train_preds = model_lgb.predict(X_tr, num_iteration=model_lgb.best_iteration)
val_preds = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)

train_auc_lgb = roc_auc_score(y_tr, train_preds)
val_auc_lgb = roc_auc_score(y_va, val_preds)
print(f"Train AUC: {train_auc_lgb:.5f}")
print(f"Val AUC  : {val_auc_lgb:.5f}")
print(f"Gap      : {train_auc_lgb - val_auc_lgb:.5f}")
print(f"Waktu    : {time_lgb:.1f} detik")

### LightGBM: Feature Importance

In [ ]:
fi_lgb = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_lgb.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_lgb = fi_lgb.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_lgb["feature"].iloc[::-1], top20_lgb["importance"].iloc[::-1], color="#4c72b0")
ax.set_xlabel("Gain Importance")
ax.set_title("LightGBM — Top 20 Fitur Berdasarkan Nilai Gain")
plt.tight_layout()
Path("../data/metadata").mkdir(parents=True, exist_ok=True)
plt.savefig("../data/metadata/lgbm_feature_importance_top20.png")
plt.show()

### XGBoost: 5-Fold Cross Validation

In [ ]:
xgb_scores = []
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "verbosity": 0,
    "random_state": 42
}

start = time.perf_counter()
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_va, label=y_va)

    early_stop = xgb.callback.EarlyStopping(rounds=30, save_best=True, metric_name="auc", data_name="val")

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=200,
        evals=[(dtrain, "train"), (dval, "val")],
        callbacks=[early_stop],
        verbose_eval=False
    )

    preds = model.predict(dval)
    xgb_scores.append(roc_auc_score(y_va, preds))

elapsed_xgb = time.perf_counter() - start
mean_xgb = np.mean(xgb_scores)
std_xgb = np.std(xgb_scores)
print(f"Rata-rata AUC: {mean_xgb:.5f} ± {std_xgb:.5f}")
print(f"Skor tiap lipatan (fold): {[round(s, 5) for s in xgb_scores]}")

### XGBoost: Pelatihan Model Final

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_va, label=y_va)
early_stop = xgb.callback.EarlyStopping(rounds=30, save_best=True, metric_name="auc", data_name="val")

start = time.perf_counter()
model_xgb = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, "train"), (dval, "val")],
    callbacks=[early_stop],
    verbose_eval=False
)
time_xgb = time.perf_counter() - start

train_preds = model_xgb.predict(dtrain)
val_preds = model_xgb.predict(dval)

train_auc_xgb = roc_auc_score(y_tr, train_preds)
val_auc_xgb = roc_auc_score(y_va, val_preds)
print(f"Train AUC: {train_auc_xgb:.5f}")
print(f"Val AUC  : {val_auc_xgb:.5f}")
print(f"Gap      : {train_auc_xgb - val_auc_xgb:.5f}")
print(f"Waktu    : {time_xgb:.1f} detik")

### XGBoost: Feature Importance

In [ ]:
score_dict = model_xgb.get_score(importance_type="gain")
fi_xgb = pd.DataFrame({
    "feature": feature_cols,
    "importance": [score_dict.get(f, 0.0) for f in feature_cols]
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_xgb = fi_xgb.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_xgb["feature"].iloc[::-1], top20_xgb["importance"].iloc[::-1], color="#dd8452")
ax.set_xlabel("Gain Importance")
ax.set_title("XGBoost — Top 20 Fitur Berdasarkan Nilai Gain")
plt.tight_layout()
plt.savefig("../data/metadata/xgb_feature_importance_top20.png")
plt.show()

### CatBoost: 5-Fold Cross Validation

In [ ]:
cb_scores = []
cb_params = {
    "iterations": 200,
    "learning_rate": 0.1,
    "depth": 6,
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False
}

start = time.perf_counter()
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr)
    val_pool = Pool(X_va, label=y_va)

    model = CatBoostClassifier(**cb_params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=30)

    preds = model.predict_proba(val_pool)[:, 1]
    cb_scores.append(roc_auc_score(y_va, preds))

elapsed_cb = time.perf_counter() - start
mean_cb = np.mean(cb_scores)
std_cb = np.std(cb_scores)
print(f"Rata-rata AUC: {mean_cb:.5f} ± {std_cb:.5f}")
print(f"Skor tiap lipatan (fold): {[round(s, 5) for s in cb_scores]}")

### CatBoost: Pelatihan Model Final

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
train_pool = Pool(X_tr, label=y_tr)
val_pool = Pool(X_va, label=y_va)

start = time.perf_counter()
model_cb = CatBoostClassifier(**cb_params)
model_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=30)
time_cb = time.perf_counter() - start

train_preds = model_cb.predict_proba(train_pool)[:, 1]
val_preds = model_cb.predict_proba(val_pool)[:, 1]

train_auc_cb = roc_auc_score(y_tr, train_preds)
val_auc_cb = roc_auc_score(y_va, val_preds)
print(f"Train AUC: {train_auc_cb:.5f}")
print(f"Val AUC  : {val_auc_cb:.5f}")
print(f"Gap      : {train_auc_cb - val_auc_cb:.5f}")
print(f"Waktu    : {time_cb:.1f} detik")

### CatBoost: Feature Importance

In [ ]:
fi_cb = pd.DataFrame({
    "feature": feature_cols,
    "importance": model_cb.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

top20_cb = fi_cb.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_cb["feature"].iloc[::-1], top20_cb["importance"].iloc[::-1], color="#55a868")
ax.set_xlabel("Tingkat Kepentingan Fitur")
ax.set_title("CatBoost — Top 20 Fitur Berdasarkan Nilai Importance")
plt.tight_layout()
plt.savefig("../data/metadata/cb_feature_importance_top20.png")
plt.show()

### Perbandingan Hasil Cross-Validation

In [ ]:
cv_results = {
    "LightGBM": lgb_scores,
    "XGBoost": xgb_scores,
    "CatBoost": cb_scores
}
cv_df = pd.DataFrame(cv_results)
print(cv_df.describe().transpose())

### Visualisasi: Boxplot Perbandingan Model

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=cv_df, ax=ax)
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Perbandingan Model — Skor Cross-Validation")
plt.show()

### Perbandingan Performa Model Final

In [ ]:
comparison = pd.DataFrame([
    {"model": "LightGBM", "val_auc": val_auc_lgb, "train_auc": train_auc_lgb, "gap": train_auc_lgb - val_auc_lgb, "time": time_lgb},
    {"model": "XGBoost", "val_auc": val_auc_xgb, "train_auc": train_auc_xgb, "gap": train_auc_xgb - val_auc_xgb, "time": time_xgb},
    {"model": "CatBoost", "val_auc": val_auc_cb, "train_auc": train_auc_cb, "gap": train_auc_cb - val_auc_cb, "time": time_cb},
])
print(comparison.to_string(index=False))